In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm

c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose your file

In [2]:
filename = "2021QCCA1675"
split = "dev"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"

with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

### System prompt loading

In [3]:
prompt_filename = "coref_long.txt"
with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)



system_prompt used :  coref_long.txt


### Assistant loading

In [4]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])

In [5]:
class ReferenceProfile:
    def __init__(self, doctype, jurisdiction:str=None, main_title=None):
        self.doc_type = doctype
        self.jurisdiction = jurisdiction
        self.main_title = main_title
        self.alternative_titles = []
        self.citations = []
        self.fragments_mentioned = []
        self.authors = [] #Only for secondary sources

    def add_alternative_title(self, title):
        self.alternative_titles.append(title)
    
    def add_citation(self, citation):
        self.citations.append(citation)
    
    def add_fragment_mentioned(self, fragment):
        self.fragments_mentioned.append(fragment)
    
    def add_author(self, author):
        self.authors.append(author)

    def __str__(self):
        return f"ReferenceProfile(main_title={self.main_title}, doc_type={self.doc_type}, jurisdiction={self.jurisdiction}, alternative_titles={self.alternative_titles}, citations={self.citations}, fragments_mentioned={self.fragments_mentioned}, authors={self.authors})"


class ReferenceProfileList:
    def __init__(self):
        self.profiles = []
    
    def add_profile(self, profile: ReferenceProfile):
        self.profiles.append(profile)

    def update(self, parsed:str):
        # Parse the LLM output and update the reference profiles accordingly
        pass
    

### Chunking with the chunker

In [6]:
chunker = "paragraph"  # "paragraph" | "sentence"
from configs.config import CHUNK_CACHE_DIR

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(cache_dir=f"{CHUNK_CACHE_DIR}/annotated", method=chunker, split=split, filename=filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp, annotated=True
    )

else:
    token_chunks = load_cache(cache_dir=f"{CHUNK_CACHE_DIR}/annotated", method=chunker, split=split, filename=filename)




In [7]:

from bs4 import BeautifulSoup


def extract_parent_level_annotations(html_content: str) -> dict:
    """
    Returns {'decision': [...], 'legislation': [...], 'secondary sources': [...]}
    Each entry is a list of annotation dicts with keys:
      full_html, docid, uri, text_content, direct_sublabels, all_sublabels.
    """
    soup = BeautifulSoup(html_content, "html.parser")
    parent_labels = soup.find_all(
        ["manual_label", "auto_label"], attrs={"parent": ""}
    )
    annotations = {"decision": [], "legislation": [], "secondary sources": []}

    idx = 0
    for label in parent_labels:
        labelname = label.get("labelname", "")
        if labelname not in annotations:
            continue
        direct = [sl.get("labelname", "")
                  for sl in label.find_all(["manual_label", "auto_label"],
                                           recursive=False)]
        all_sl = [sl.get("labelname", "")
                  for sl in label.find_all(["manual_label", "auto_label"])]
        annotations[labelname].append({
            "full_html":       str(label),
            "docid":           label.get("docid", ""),
            "uri":             label.get("uri", ""),
            "text_content":    label.get_text(strip=True),
            "direct_sublabels": direct,
            "all_sublabels":   all_sl,
            "order": idx,
        })
        idx += 1
    return annotations

In [8]:
def get_user_input_for_coref(input_token_chunk, list_mention, reference_profile_list):
    # Return a user string that includes the decoded token chunk, the list of references identified and the list of references mentioned in the previous chunks (from the reference profile list)
    return f"""Here is a chunk of text from a legal document:\n{decode(input_token_chunk)}.

References identified in this chunk:
{'\n---\n'.join([mention["text_content"] for mention in list_mention])}

References mentioned in previous chunks:
{'\n---\n'.join(str(profile) for profile in reference_profile_list.profiles) if len(reference_profile_list.profiles) > 0 else "None"}"""

### Main processing function

In [9]:
from src.ann_extractor import extract_parent_level_annotations

In [32]:
token_chunk = token_chunks[9]
list_mention_in_current_chunk = [
    mention
    for value in extract_parent_level_annotations(decode(token_chunk)).values()
    for mention in value
]
list_mention_in_current_chunk.sort(key=lambda x: x["order"])

In [34]:
list_mention_in_current_chunk

[{'full_html': '<manual_label docid="Laicity Act" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/56lzp" verified="false"><manual_label labelname="title" parent="legislation" style="background-color: rgb(147, 196, 125); color: black;" titletype="alias" verified="false">Laicity\nAct</manual_label></manual_label>',
  'docid': 'Laicity Act',
  'uri': 'https://canlii.ca/t/56lzp',
  'text_content': 'Laicity\nAct',
  'direct_sublabels': ['title'],
  'all_sublabels': ['title'],
  'order': 0},
 {'full_html': '<manual_label docid="Laicity Act" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/56lzp" verified="false"><manual_label labelname="title" parent="legislation" style="background-color: rgb(147, 196, 125); color: black;" titletype="alias" verified="false">Laicity Act</manual_label> </manual_label>',
  'docid': 'Laicity Act',
  'uri': 'https://canlii.c

In [13]:
reference_profile_list = ReferenceProfileList()

In [14]:
cfg = LabelTransformConfig(
    keep_labels=["decision", "legislation", "secondary sources"],
    use_simplified=True,
    keep_attributes=["labelname"]
)
user_input =  get_user_input_for_coref(input_token_chunk=prepare_label_tokens(token_chunk, cfg=cfg), list_mention=list_mention_in_current_chunk,reference_profile_list=reference_profile_list)


In [15]:
print(user_input)

Here is a chunk of text from a legal document:
[16] The applicants’ second claim is that the <legislation>Laicity
Act</legislation> will seriously harm the integration and educational success of
students, notably their ethnic and religious minority students, who will be
deprived of the full benefits of a pedagogical approach emphasizing the
acceptance and celebration of religious diversity.[31] They
support their claim by invoking a number of items of evidence that were adduced
at trial and that point to the benefits of religious diversity among
administrative and teaching staff, as well as the fact that no other party — not
even the Attorney General — adduced any evidence to the contrary. [17] However, even assuming that the evidence
supports the findings made by Blanchard J., it sheds little light on the extent
to which the perpetuation of that pedagogical approach will depend on
recruitment or promotion decisions which the <legislation>Laicity Act </legislation>will prevent
the Angl

In [ ]:
from src.models import get_messages
from tqdm import tqdm

reference_profile_list = ReferenceProfileList()
processed_chunks = []
for token_chunk in tqdm(token_chunks):

    list_mention_in_current_chunk = [
        mention
        for value in extract_parent_level_annotations(decode(token_chunk)).values()
        for mention in value
    ]
    list_mention_in_current_chunk.sort(key=lambda x: x["order"])

    ############ Create the user message
    user_input =  get_user_input_for_coref(input_token_chunk=token_chunk, list_mention=list_mention_in_current_chunk,reference_profile_list=reference_profile_list)

    message = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=assistant.has_system_role)

    ############ Generate
    generated = assistant.generate(messages=message)

    parsed_generated = parse_coref_output(generated)
    # parsed_generated : {text_mention: main_title_group}

    # Little conversion {text_mention: main_title_group} -> {Mention object: main_title_group}
    converted_parsed_generated = {}
    for text_mention in parsed_generated.keys():
        mention_obj = next((mention for mention in list_mention_in_current_chunk if mention.text == text_mention), None) #might be too strick, we may consider 
        if mention_obj:
            converted_parsed_generated[mention_obj] = parsed_generated[text_mention]

    ############ Update the reference profile list
    reference_profile_list.update(converted_parsed_generated)

    ############ Add the docid attribute to the mention objects in the current chunk

